# Week 7: Document Question Answering System (RAG)

**Retrieval-Augmented Generation (RAG) Pipeline**

This notebook implements a simple RAG system that answers questions from a custom document (PDF or text file) by combining:
1. **Retrieval** — finding the most relevant chunks of text using embeddings + vector similarity
2. **Augmentation** — adding retrieved chunks as context
3. **Generation** — producing a grounded answer from that context

Two modes are supported for each component so this runs **out of the box with zero downloads**, with an easy upgrade path to higher-quality models when internet/GPU access is available:

| Component | Lightweight (default) | Upgrade |
|---|---|---|
| Embeddings | TF-IDF (scikit-learn) | `sentence-transformers` (semantic embeddings) |
| Generation | Extractive (best-matching chunk) | `flan-t5-base` via `transformers` (abstractive answers) |


## 0. Install Dependencies

Only `scikit-learn`, `numpy`, and `pypdf` are required for the default lightweight mode.

In [12]:
# Core dependencies (lightweight mode — always required)
%pip install -q scikit-learn numpy pypdf

# Optional upgrade — uncomment if you have internet access and want higher quality results
# %pip install -q sentence-transformers transformers torch

## 1. Imports

In [13]:
import os
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Document Ingestion

Loads a document from disk. Supports `.txt` and `.pdf` files — this is where you'd point it at your resume, notes, or research papers.

In [14]:
def load_document(file_path: str) -> str:
    """Load a .txt or .pdf file and return its raw text."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    elif ext == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text

    else:
        raise ValueError(f"Unsupported file type: {ext}. Use .txt or .pdf")

## 3. Text Chunking

Splits the document into overlapping word-based chunks. Chunking improves retrieval accuracy because the system can pinpoint the exact passage that answers a question, instead of retrieving an entire document.

In [15]:
def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20):
    """Split text into overlapping word-based chunks."""
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap  # move forward, keeping overlap for continuity

    return chunks

## 4. Embedding Creation

Converts each text chunk into a numerical vector.

- `mode="tfidf"` — fully offline, no downloads, good enough for keyword-driven questions on a single document (default here).
- `mode="sentence-transformer"` — captures semantic meaning (e.g. matches "car" with "automobile"), needs `sentence-transformers` + internet access to download the model.

In [16]:
class Embedder:
    def __init__(self, mode: str = "tfidf"):
        self.mode = mode
        if mode == "sentence-transformer":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
        else:
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.fitted = False

    def fit(self, texts):
        if self.mode == "tfidf":
            self.vectorizer.fit(texts)
            self.fitted = True

    def encode(self, texts):
        if self.mode == "sentence-transformer":
            return np.array(self.model.encode(texts))
        else:
            if not self.fitted:
                raise RuntimeError("Call .fit(texts) before .encode() in tfidf mode")
            return self.vectorizer.transform(texts).toarray()

## 5. Vector Store

A minimal in-memory vector database. For larger document collections you'd swap this for **FAISS**, **Chroma**, or **Pinecone**, but cosine similarity over a NumPy array is enough for a single document.

In [17]:
class VectorStore:
    """A minimal in-memory vector store using cosine similarity search."""

    def __init__(self):
        self.embeddings = None
        self.chunks = []

    def add(self, chunks, embeddings):
        self.chunks = chunks
        self.embeddings = embeddings

    def search(self, query_embedding, top_k=3):
        sims = cosine_similarity(query_embedding, self.embeddings)[0]
        top_indices = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[i], float(sims[i])) for i in top_indices]

## 6. Answer Generation

Turns retrieved chunks + the question into a final answer.

- `mode="extractive"` — returns the best-matching retrieved passage directly (no model download, always works).
- `mode="flan-t5"` — feeds the context + question into a small open-source LLM (`google/flan-t5-base`) for an abstractive, more natural-sounding answer. Needs `transformers` + `torch` + internet access.

In [18]:
class Generator:
    def __init__(self, mode: str = "extractive"):
        self.mode = mode
        if mode == "flan-t5":
            from transformers import pipeline
            self.pipe = pipeline("text2text-generation", model="google/flan-t5-base")

    def generate(self, query: str, retrieved_chunks) -> str:
        context = "\n\n".join(chunk for chunk, _ in retrieved_chunks)

        if self.mode == "flan-t5":
            prompt = (
                f"Answer the question using only the context below. "
                f"If the answer is not in the context, say you don't know.\n\n"
                f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
            )
            result = self.pipe(prompt, max_length=200)
            return result[0]["generated_text"]

        # Extractive fallback: return the single best-matching chunk
        best_chunk, score = retrieved_chunks[0]
        return best_chunk

## 7. Full RAG Pipeline

Wires ingestion → chunking → embedding → vector store → retrieval → generation into one class.

In [19]:
class RAGPipeline:
    def __init__(self, embedder_mode="tfidf", generator_mode="extractive",
                 chunk_size=120, overlap=20):
        self.embedder = Embedder(mode=embedder_mode)
        self.generator = Generator(mode=generator_mode)
        self.store = VectorStore()
        self.chunk_size = chunk_size
        self.overlap = overlap

    def build_index(self, file_path: str):
        raw_text = load_document(file_path)
        chunks = chunk_text(raw_text, self.chunk_size, self.overlap)
        self.embedder.fit(chunks)
        embeddings = self.embedder.encode(chunks)
        self.store.add(chunks, embeddings)
        print(f"Indexed {len(chunks)} chunks from '{file_path}'")

    def answer(self, query: str, top_k: int = 3, show_sources: bool = True):
        query_embedding = self.embedder.encode([query])
        retrieved = self.store.search(query_embedding, top_k=top_k)
        answer_text = self.generator.generate(query, retrieved)

        if show_sources:
            print(f"\nQ: {query}")
            print(f"A: {answer_text}\n")
            print("Top retrieved chunks:")
            for i, (chunk, score) in enumerate(retrieved, 1):
                print(f"  [{i}] (score={score:.3f}) {chunk[:120]}...")
        return answer_text, retrieved

## 8. Demo: Build the Index

A sample document about RAG itself is included below so the notebook runs immediately. **Replace `sample_document.txt` with your own PDF or text file** (resume, notes, research paper) to try it on your own data — that's the whole point of RAG.

In [20]:
rag = RAGPipeline(embedder_mode="tfidf", generator_mode="extractive")
rag.build_index("sample_document.txt")

Indexed 5 chunks from 'sample_document.txt'


## 9. Ask Questions

In [21]:
questions = [
    "What is RAG and why does it matter?",
    "What is text chunking and why is it used?",
    "What are some common applications of RAG systems?",
]

for q in questions:
    rag.answer(q)


Q: What is RAG and why does it matter?
A: Retrieval-Augmented Generation: An Overview Retrieval-Augmented Generation, or RAG, is a technique that combines information retrieval with text generation. Instead of relying purely on the internal knowledge stored in a language model's parameters, a RAG system first retrieves relevant pieces of text from an external knowledge source, such as a collection of documents, and then uses that retrieved text as context for generating an answer. Why RAG Matters Large language models are trained on a fixed snapshot of data and can produce outdated or incorrect information when asked about topics outside their training data. RAG solves this by grounding the model's responses in real, up to date, or domain specific documents. This makes RAG especially useful for private company data, personal

Top retrieved chunks:
  [1] (score=0.323) Retrieval-Augmented Generation: An Overview Retrieval-Augmented Generation, or RAG, is a technique that combines inform

## 10. Using Your Own Document

```python
rag = RAGPipeline(embedder_mode="tfidf", generator_mode="extractive")
rag.build_index("your_resume.pdf")   # or notes.txt, research_paper.pdf, etc.
rag.answer("What projects has this person worked on?")
```

## 11. Upgrading to Semantic Embeddings + an LLM

On Colab or a machine with internet access, swap the modes for higher quality:

```python
# pip install sentence-transformers transformers torch

rag = RAGPipeline(embedder_mode="sentence-transformer", generator_mode="flan-t5")
rag.build_index("your_document.pdf")
rag.answer("What is the main idea of the document?")
```

`sentence-transformer` mode captures semantic meaning (matches paraphrases, not just keywords). `flan-t5` mode writes a fluent answer in its own words instead of returning a raw chunk.

## Key Learnings
- How retrieval and generation combine to ground LLM answers in real documents
- Why chunking with overlap improves retrieval precision
- How embeddings + cosine similarity power semantic/keyword search
- Designing a pipeline that degrades gracefully (works offline, upgrades cleanly online)

## Conclusion
This notebook builds a working, minimal RAG system: it ingests a document, chunks it, embeds the chunks, retrieves the most relevant ones for a query, and generates a grounded answer — the same pattern used in production chatbots, enterprise search, and knowledge assistants, just at beginner scale.
